# Supplementary — automatic-control operculum pair figures

Publication layout copied from the D0079 LAI7–8 / LFMI2–3 case figure. This notebook draws the other operculum pairs that already have saved before-orthogonalization HGA. It does not edit that case notebook, recompute orthogonalization, or change tests.

Each figure is one row, 18 cm by 4.398 cm: oblique 3D brain, one superior-up native T1, and Stimulus / Delay / Response HGA for the two bipolar sites.

In [1]:
from pathlib import Path
import sys, json, tempfile
import matplotlib
matplotlib.use('Agg')
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.project-root').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.text import Annotation
for path in (Path(sys.prefix) / 'fonts').glob('arial*.ttf'):
    font_manager.fontManager.addfont(str(path))
plt.rcParams.update({
    'svg.fonttype': 'none', 'font.family': 'sans-serif', 'font.sans-serif': ['Arial'],
    'font.size': 7, 'axes.linewidth': .75, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
})
cm = 1 / 2.54
TOPIC = 'operculum_pairs'
IMG_DIR = ROOT / 'img/supplementary' / TOPIC
IMG_DIR.mkdir(parents=True, exist_ok=True)
OPER_DIR = ROOT / 'results/supplementary/operculum'
MANUAL_PAIR = 'LexicalDelay__D0079_LAI7-8__D0079_LFMI2-3'
# load_subject writes an image-source receipt. Keep that off the frozen anatomy directory.
import src.supplementary.operculum_anatomy as anatomy
anatomy.OUT = Path(tempfile.mkdtemp(prefix='operculum_pair_sources_'))
from src.main_analysis import RESPONSE_WINDOWS, ROI_COLORS
from src.supplementary.operculum_anatomy import (
    COLORS, load_subject, pair_points, full_shank_contacts, render_surface,
    sample_plane, INSULA_NAMES, MRI_PROJECTION_TOL_MM)
PHASE_WINDOWS = {'Stimulus': (0.0, 0.6), 'Delay': (0.0, 1.0), 'Response': (-0.5, 0.5)}
assert {phase: RESPONSE_WINDOWS[phase.lower()] for phase in PHASE_WINDOWS} == PHASE_WINDOWS
REQUIRED_CONDITIONS = ['AIC_before', 'AIC_orthogonal', 'FO_before', 'FO_orthogonal']


In [2]:
def bipolar_name(site_id):
    """D0079_LAI7-8 -> LAI7–8. The en dash matches the publication labels."""
    return site_id.split('_', 1)[1].replace('-', '–')

candidates = []
skipped = []
for path in sorted((OPER_DIR / 'pairs').glob('*/completed.json')):
    receipt = json.loads(path.read_text())
    pair_id = receipt['pair_id']
    if pair_id == MANUAL_PAIR:
        continue
    directory = path.parent
    reasons = []
    phase_paths = {phase: directory / f'{phase}.npz' for phase in PHASE_WINDOWS}
    for phase, npz in phase_paths.items():
        if not npz.is_file():
            reasons.append(f'missing {phase}.npz')
    summary_path = directory / 'response_summary.csv'
    if not summary_path.is_file():
        reasons.append('missing response_summary.csv')
    if not reasons:
        summary = pd.read_csv(summary_path)
        for phase, (lo, hi) in PHASE_WINDOWS.items():
            part = summary.loc[summary.phase.eq(phase)]
            if part.empty or not (np.allclose(part.window_start, lo) and np.allclose(part.window_end, hi)):
                reasons.append(f'{phase} window does not match the publication windows')
        for phase, npz in phase_paths.items():
            with np.load(npz) as data:
                conditions = [str(value) for value in data['conditions']]
                if conditions != REQUIRED_CONDITIONS:
                    reasons.append(f'{phase} conditions are {conditions}')
                for key in ('zscore', 'times', 'mask', 'pvals'):
                    if key not in data.files:
                        reasons.append(f'{phase} npz has no {key}')
    info = receipt.get('pair')
    if not isinstance(info, dict) or not {'subject', 'hemisphere', 'task', 'insula_site', 'operculum_site'} <= set(info):
        reasons.append('completed.json has no native pair record')
    elif not reasons:
        try:
            subject = load_subject(info['subject'], info['hemisphere'])
            points = pair_points(info, subject)
            full_shank_contacts(subject, points)
        except Exception as exc:
            reasons.append(f'native anatomy unavailable: {exc}')
    if reasons:
        skipped.append(dict(pair_id=pair_id, reason='; '.join(reasons)))
    else:
        candidates.append(dict(pair_id=pair_id, directory=directory, info=info))

index_ids = set(pd.read_csv(OPER_DIR / 'pair_index.csv').pair_id)
found_ids = {item['pair_id'] for item in candidates} | {item['pair_id'] for item in skipped}
assert found_ids == index_ids, (sorted(found_ids ^ index_ids))
assert MANUAL_PAIR not in found_ids
print(f'{len(candidates)} other pairs with saved HGA and native anatomy')
for item in candidates:
    info = item['info']
    print(f"  {item['pair_id']}  {bipolar_name(info['insula_site'])} / {bipolar_name(info['operculum_site'])}")
if skipped:
    print(f'{len(skipped)} skipped')
    for item in skipped:
        print(f"  {item['pair_id']}: {item['reason']}")
else:
    print('0 skipped')


15 other pairs with saved HGA and native anatomy
  LexicalDelay__D0079_LAI1-2__D0079_LFMI7-8  LAI1–2 / LFMI7–8
  LexicalDelay__D0079_LAI2-3__D0079_LFMI7-8  LAI2–3 / LFMI7–8
  LexicalDelay__D0079_LAI3-4__D0079_LFMI7-8  LAI3–4 / LFMI7–8
  LexicalDelay__D0079_LAI7-8__D0079_LFMI7-8  LAI7–8 / LFMI7–8
  LexicalDelay__D0094_LFAI2-3__D0094_LFAI8-9  LFAI2–3 / LFAI8–9
  LexicalDelay__D0094_LIA3-4__D0094_LFAI8-9  LIA3–4 / LFAI8–9
  PhonemeSequence__D0057_RAI3-4__D0057_RAMT9-10  RAI3–4 / RAMT9–10
  SentenceRep__D0032_LAI1-2__D0032_LAI6-7  LAI1–2 / LAI6–7
  SentenceRep__D0032_LAI2-3__D0032_LAI6-7  LAI2–3 / LAI6–7
  SentenceRep__D0057_RAI3-4__D0057_RAMT9-10  RAI3–4 / RAMT9–10
  SentenceRep__D0064_LAI3-4__D0064_LAIF6-7  LAI3–4 / LAIF6–7
  SentenceRep__D0064_LAI4-5__D0064_LAIF6-7  LAI4–5 / LAIF6–7
  SentenceRep__D0064_LAI5-6__D0064_LAIF6-7  LAI5–6 / LAIF6–7
  SentenceRep__D0064_LAI6-7__D0064_LAIF6-7  LAI6–7 / LAIF6–7
  SentenceRep__D0102_RFAI1-2__D0102_RFAI3-4  RFAI1–2 / RFAI3–4
0 skipped


In [3]:
def near_coronal_pair_plane(points):
    """Plane through both bipolar midpoints, with the smallest tilt from coronal.

    The normal is the anterior–posterior axis after removing its component along
    the midpoint–midpoint vector, so it stays perpendicular to that vector.
    Horizontal in the slice stays as close as possible to left–right; vertical
    stays superior. Contacts off the plane are projected, not used to tilt it.
    """
    mids = []
    for region in ('AIC', 'FO'):
        row = points.loc[points.region.eq(region) & points.role.eq('midpoint')].iloc[0]
        mids.append(row[['x', 'y', 'z']].to_numpy(dtype=float))
    span = mids[1] - mids[0]
    span /= np.linalg.norm(span)
    coronal = np.array([0., 1., 0.])
    normal = coronal - np.dot(coronal, span) * span
    normal /= np.linalg.norm(normal)
    if normal[1] > 0:
        normal = -normal
    superior = np.array([0., 0., 1.])
    axis2 = superior - np.dot(superior, normal) * normal
    axis2 /= np.linalg.norm(axis2)
    if axis2[2] < 0:
        axis2 = -axis2
    right = np.array([1., 0., 0.])
    axis1 = right - np.dot(right, normal) * normal
    axis1 -= np.dot(axis1, axis2) * axis2
    axis1 /= np.linalg.norm(axis1)
    if axis1[0] < 0:
        axis1 = -axis1
    origin = np.dot(mids[0], normal) * normal
    tilt = float(np.degrees(np.arccos(np.clip(abs(float(normal[1])), -1., 1.))))
    return origin, axis1, axis2, normal, tilt

def trial_mean_sem(trials):
    """Trial mean and SEM. SEM is nanstd (ddof 1) over finite trials / sqrt(n)."""
    count = np.isfinite(trials).sum(axis=0).astype(float)
    mean = np.nanmean(trials, axis=0)
    with np.errstate(invalid='ignore', divide='ignore'):
        sem = np.nanstd(trials, axis=0, ddof=1) / np.sqrt(count)
    sem[count < 2] = np.nan
    return mean, sem

def significant_times(data, idx, times, lo, hi):
    """Cluster samples with p < 0.05 that also fall inside the phase window."""
    cluster = np.asarray(data['mask'][idx], dtype=bool) & (np.asarray(data['pvals'][idx]) < 0.05)
    inside = (times >= lo) & (times < hi)
    for key in ('window_mask', 'significant_in_window'):
        if key in data.files and np.shape(data[key]) == np.shape(data['mask']):
            cluster = cluster & np.asarray(data[key][idx], dtype=bool)
            break
    return cluster & inside

def draw_publication_mri(ax, subject, points, pair_names):
    """Superior-up T1 through both midpoints. One label per pair."""
    origin, axis1, axis2, normal, tilt = near_coronal_pair_plane(points)
    scale, t1 = sample_plane(subject, 'mri', origin, axis1, axis2, half=90., spacing=.5)
    _, atlas = sample_plane(subject, 'aparc', origin, axis1, axis2, half=90., spacing=.5, order=0)
    extent = [scale[0], scale[-1], scale[0], scale[-1]]
    ax.imshow(t1, cmap='gray', origin='lower', extent=extent,
              vmin=subject['mri_limits'][0], vmax=subject['mri_limits'][1], interpolation='none')
    prefix = f'ctx_{subject["hemi"]}_'
    for label_region in ('AIC', 'FO'):
        if label_region == 'AIC':
            codes = [code for code, name in subject['lut'].items()
                     if name.startswith(prefix) and any(n in name for n in INSULA_NAMES)]
        else:
            codes = [code for code, name in subject['lut'].items()
                     if name.startswith(prefix) and 'G_front_inf-Opercular' in name]
        mask = np.isin(atlas, codes)
        if mask.any() and not mask.all():
            ax.imshow(np.ma.masked_where(~mask, np.ones(mask.shape)),
                      cmap=ListedColormap([COLORS[label_region]]), origin='lower', extent=extent,
                      alpha=.18, interpolation='none', vmin=0, vmax=1)
            ax.contour(scale, scale, mask, levels=[.5], colors=[COLORS[label_region]],
                       linewidths=.65, alpha=.85)
    label_offsets = {
        'AIC': (np.array([16., -16.]), 'left', 'center'),
        'FO': (np.array([-16., 16.]), 'right', 'center'),
    }
    for region in ('AIC', 'FO'):
        frame = (points.loc[points.region.eq(region)].set_index('role')
                 .loc[['contact_1', 'midpoint', 'contact_2']].reset_index())
        xyz = frame[['x', 'y', 'z']].to_numpy(dtype=float)
        depth = (xyz - origin) @ normal
        assert np.max(np.abs(depth)) <= MRI_PROJECTION_TOL_MM, (region, np.round(depth, 3))
        projected = np.column_stack([(xyz - origin) @ axis1, (xyz - origin) @ axis2])
        ax.plot(projected[[0, 2], 0], projected[[0, 2], 1], c=COLORS[region], lw=.7, zorder=7)
        for i in range(3):
            x, y = projected[i]
            ax.scatter(x, y, s=9 if i == 1 else 6, marker='D' if i == 1 else 'o',
                       facecolors='none', edgecolors=COLORS[region], linewidths=.6, zorder=8)
        offset, ha, va = label_offsets[region]
        ax.annotate(pair_names[region], projected[1], xytext=projected[1] + offset,
                    textcoords='data', ha=ha, va=va, fontsize=6.5, color=COLORS[region],
                    arrowprops=dict(arrowstyle='-', color=COLORS[region], lw=.6),
                    zorder=9)
    ax.plot([-78, -58], [-76, -76], c='white', lw=1)
    ax.text(-68, -83, '20 mm', color='white', ha='center', fontsize=6)
    for axis, positive, negative in [(axis1, (.98, .5), (.02, .5)), (axis2, (.5, .98), (.5, .02))]:
        dominant = int(np.argmax(np.abs(axis)))
        pos, neg = ['R', 'A', 'S'][dominant], ['L', 'P', 'I'][dominant]
        if axis[dominant] < 0:
            pos, neg = neg, pos
        for name, where in ((pos, positive), (neg, negative)):
            ax.text(*where, name, transform=ax.transAxes, color='white', fontsize=6,
                    ha='center', va='center',
                    bbox=dict(facecolor='black', alpha=.5, edgecolor='none', pad=.5))
    ax.set_xlim(-90, 90)
    ax.set_ylim(-90, 90)
    ax.set_aspect('equal')
    ax.axis('off')
    return tilt

def listed_pixels_per_mm(points, pixels):
    """Publication scale when the six pair points span 3D."""
    xyz = points[['x', 'y', 'z']].to_numpy(dtype=float)
    pix = np.asarray(pixels, dtype=float)
    design = np.column_stack([xyz, np.ones(len(xyz))])
    coef, _, rank, _ = np.linalg.lstsq(design, pix, rcond=None)
    residual = float(np.max(np.linalg.norm(design @ coef - pix, axis=1)))
    jacobian = coef[:3].T
    singular = np.linalg.svd(jacobian, compute_uv=False)
    if rank < 4:
        return None
    assert residual < 0.5, residual
    assert singular[1] > 0 and abs(singular[0] - singular[1]) / singular[0] < 0.02, singular
    horizontal = np.linalg.lstsq(jacobian, np.array([1.0, 0.0]), rcond=None)[0]
    return 1.0 / float(np.linalg.norm(horizontal))

def camera_pixels_per_mm(shanks, image):
    """Same parallel-camera scale render_surface uses for zoom=True.

    VTK parallel_scale is half the window height in world units. On a pair whose
    points span 3D this matches listed_pixels_per_mm. Same-shaft pairs are
    coplanar, so that fit is not defined and this camera scale is used instead.
    """
    all_xyz = shanks[['x', 'y', 'z']].to_numpy(dtype=float)
    parallel_scale = max(22., np.linalg.norm(np.ptp(all_xyz, axis=0)) / 2 + 12.)
    height = float(image.shape[0])
    assert height == 850, image.shape
    return height / (2.0 * parallel_scale)

def parallel_pixels_per_mm(points, pixels, shanks, image):
    fitted = listed_pixels_per_mm(points, pixels)
    camera = camera_pixels_per_mm(shanks, image)
    if fitted is None:
        return camera
    assert abs(fitted - camera) / camera < 1e-6, (fitted, camera)
    return fitted

def draw_mm_bar(ax, image, px_per_mm):
    bar = 20.0 * px_per_mm
    x0, y0 = 28.0, float(image.shape[0]) - 34.0
    assert x0 + bar < image.shape[1] - 16, bar
    ax.plot([x0, x0 + bar], [y0, y0], color='white', lw=2.4, solid_capstyle='butt', zorder=6)
    ax.plot([x0, x0 + bar], [y0, y0], color='black', lw=1.0, solid_capstyle='butt', zorder=7)
    ax.annotate('20 mm', xy=(x0 + bar / 2, y0), xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=6, color='black',
                bbox=dict(facecolor='white', edgecolor='none', alpha=.85, pad=.2), zorder=7)

def draw_publication_brain(ax, zoom, subject, points, shanks, pair_names):
    """Full native shanks. Bipolar pair names only."""
    assert zoom
    image, pixels = render_surface(subject, points, shanks, zoom)
    ax.imshow(image)
    ax.axis('off')
    px_per_mm = parallel_pixels_per_mm(points, pixels, shanks, image)
    anchors = {}
    for region in ('AIC', 'FO'):
        inds = np.flatnonzero(points.region.eq(region))
        xy = pixels[inds]
        ax.plot(xy[[0, 2], 0], xy[[0, 2], 1], c=COLORS[region], lw=.7, zorder=4)
        ax.scatter(xy[[0, 2], 0], xy[[0, 2], 1], s=12, marker='o', c=COLORS[region],
                   edgecolors='white', linewidths=.25, zorder=5)
        anchors[region] = (xy, pair_names[region], inds)
    aic_mid, fo_mid = anchors['AIC'][0][1], anchors['FO'][0][1]
    delta = fo_mid - aic_mid
    delta = delta / np.linalg.norm(delta)
    perp = np.array([-delta[1], delta[0]])
    for region, sign, along in (('AIC', -1.0, -10.0), ('FO', 1.0, 10.0)):
        xy, label, inds = anchors[region]
        text_at = xy[1] + sign * perp * 48 + delta * along
        text_at = np.clip(text_at, [24, 24], [image.shape[1] - 24, image.shape[0] - 24])
        ax.annotate(label, xy=xy[1], xytext=text_at, textcoords='data',
                    ha='center', va='center', fontsize=6.5, color=COLORS[region],
                    arrowprops=dict(arrowstyle='-', color=COLORS[region], lw=.6),
                    zorder=6)
    draw_mm_bar(ax, image, px_per_mm)
    ax.set_title('Oblique', fontsize=7, pad=2)

def save_publication(fig, pair_id):
    target = IMG_DIR / f'{pair_id}__hga_coronal_pub.svg'
    assert 'operculum_D0079_case' not in target.parts
    assert target.name != 'LexicalDelay__D0079_LAI7-8__D0079_LFMI2-3__hga_coronal_pub.svg'
    fig.savefig(target, dpi=300)
    plt.close(fig)
    return target

drawn = []
for item in candidates:
    pair_id = item['pair_id']
    directory = item['directory']
    info = item['info']
    pair_names = {
        'AIC': bipolar_name(info['insula_site']),
        'FO': bipolar_name(info['operculum_site']),
    }
    phase_data = {phase: np.load(directory / f'{phase}.npz') for phase in PHASE_WINDOWS}
    native_subject = load_subject(info['subject'], info['hemisphere'])
    native_points = pair_points(info, native_subject)
    origin, _, _, normal, tilt = near_coronal_pair_plane(native_points)
    off_plane = None
    for region in ('AIC', 'FO'):
        frame = native_points.loc[native_points.region.eq(region)]
        depth = (frame[['x', 'y', 'z']].to_numpy(float) - origin) @ normal
        if np.max(np.abs(depth)) > MRI_PROJECTION_TOL_MM:
            off_plane = (region, float(np.max(np.abs(depth))))
            break
    if off_plane is not None:
        region, depth_mm = off_plane
        skipped.append(dict(pair_id=pair_id, reason=f'{region} contacts are {depth_mm:.2f} mm off the T1 plane'))
        for data in phase_data.values():
            data.close()
        continue
    shanks = full_shank_contacts(native_subject, native_points)
    channels = (
        (0, ROI_COLORS['Insula'], pair_names['AIC']),
        (2, ROI_COLORS['IFG'], pair_names['FO']),
    )
    curves = [trial_mean_sem(data['zscore'][:, idx, :]) for data in phase_data.values() for idx, _, _ in channels]
    stack = np.concatenate([np.concatenate([mean - sem, mean + sem]) for mean, sem in curves])
    stack = stack[np.isfinite(stack)]
    low, high = min(0., float(stack.min())), max(0., float(stack.max()))
    pad = max(.1, (high - low) * .12)

    scale = 18 / 22.10
    hga_cm = 3.8 * scale
    side_cm = 4.2 * scale
    label_cm, gap_hga_cm = 1.12 * scale, 0.28 * scale
    outer_gap_cm = 0.22 * scale
    left_cm, right_cm = 0.10 * scale, 0.08 * scale
    top_cm, bottom_cm = 0.50 * scale, 0.70 * scale
    hga_block_cm = label_cm + 3 * hga_cm + 2 * gap_hga_cm
    fig_w_cm = left_cm + 2 * side_cm + 2 * outer_gap_cm + hga_block_cm + right_cm
    fig_h_cm = top_cm + side_cm + bottom_cm
    assert abs(fig_w_cm - 18) < 1e-9
    assert abs(fig_h_cm - 4.398190045248869) < 1e-9
    fig = plt.figure(figsize=(fig_w_cm * cm, fig_h_cm * cm))
    left, right = left_cm / fig_w_cm, 1 - right_cm / fig_w_cm
    top, bottom = 1 - top_cm / fig_h_cm, bottom_cm / fig_h_cm
    sum_outer = 2 * side_cm + hga_block_cm
    outer_wspace = 3 * outer_gap_cm / sum_outer
    outer = fig.add_gridspec(
        1, 3, width_ratios=(side_cm, side_cm, hga_block_cm), wspace=outer_wspace,
        left=left, right=right, top=top, bottom=bottom)
    brain_ax = fig.add_subplot(outer[0, 0])
    mri_ax = fig.add_subplot(outer[0, 1])
    hga_slot = outer[0, 2].get_position(fig)
    hga_cm = (hga_slot.width * fig_w_cm - label_cm - 2 * gap_hga_cm) / 3
    hga_x = hga_slot.x0 * fig_w_cm + label_cm
    axes = []
    for col in range(3):
        rect = [hga_x / fig_w_cm, hga_slot.y0, hga_cm / fig_w_cm, hga_slot.height]
        axes.append(fig.add_axes(rect, sharex=axes[0] if col else None, sharey=axes[0] if col else None))
        hga_x += hga_cm + gap_hga_cm
    for col, (phase, data) in enumerate(phase_data.items()):
        ax = axes[col]
        times = np.asarray(data['times'], float)
        lo, hi = PHASE_WINDOWS[phase]
        ax.axvspan(lo, hi, facecolor='0.45', alpha=0.16, lw=0, zorder=0)
        for (idx, color, label), ymark in zip(channels, (0.045, 0.10)):
            mean, sem = trial_mean_sem(data['zscore'][:, idx, :])
            ax.fill_between(times, mean - sem, mean + sem, color=color, alpha=0.2,
                            linewidth=0, zorder=2)
            ax.plot(times, mean, color=color, lw=.9, zorder=3, label=label if col == 0 else None)
            sig = significant_times(data, idx, times, lo, hi)
            ax.scatter(times[sig], np.full(int(sig.sum()), ymark), marker='_', s=5, color=color,
                       transform=ax.get_xaxis_transform(), linewidths=.7, zorder=4)
        ax.axvline(0, color='0.5', lw=.5, zorder=1)
        ax.axhline(0, color='0.7', lw=.5, zorder=1)
        ax.set_xlim(-.5, 1.5)
        ax.set_ylim(low - pad, high + pad)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_title(phase, fontsize=7, pad=2)
        ax.tick_params(axis='both', labelsize=7, length=2.5, pad=1)
        if col:
            ax.tick_params(labelleft=False)
    axes[0].set_ylabel('HGA (z)', fontsize=7)
    sig_proxy = Line2D([], [], color='0.15', marker='_', linestyle='None', markersize=7, markeredgewidth=.8)
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles + [sig_proxy], labels + ['significant cluster (p < 0.05)'],
                   frameon=False, fontsize=7, loc='upper left')
    draw_publication_brain(brain_ax, zoom=True, subject=native_subject, points=native_points,
                           shanks=shanks, pair_names=pair_names)
    drawn_tilt = draw_publication_mri(mri_ax, native_subject, native_points, pair_names)
    assert abs(drawn_tilt - tilt) < 1e-9
    mri_ax.set_title('Native T1', fontsize=7, pad=2)
    fig.text((axes[0].get_position().x0 + axes[2].get_position().x1) / 2,
             0.17 / fig_h_cm, 'Time (s)', ha='center', va='center', fontsize=7)
    contact_names = set(native_points.loc[native_points.role.ne('midpoint'), 'contact'])
    panel_text = [text.get_text() for ax in (brain_ax, mri_ax) for text in ax.texts]
    assert contact_names.isdisjoint(panel_text)
    for ax in (brain_ax, mri_ax):
        for text in ax.texts:
            if isinstance(text, Annotation) and text.arrow_patch is not None and text.get_text() in pair_names.values():
                assert text.get_bbox_patch() is None, text.get_text()
    target = save_publication(fig, pair_id)
    for data in phase_data.values():
        data.close()
    drawn.append(str(target))
    print(target)

print(f'drew {len(drawn)}')
print(f'skipped {len(skipped)}')
for item in skipped:
    print(f"SKIP {item['pair_id']}: {item['reason']}")


2026-10-20 21:51:37.180 (   0.874s) [    7FB5F6E3F740]vtkXOpenGLRenderWindow.:1416  WARN| bad X server connection. DISPLAY=
libEGL warning: pci id for fd 95: 10de:26b2, driver (null)

pci id for fd 96: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 95: 10de:26b2, driver (null)

pci id for fd 96: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card8: Permission denied

2026-10-20 21:51:37.238 (   0.932s) [    7FB5F6E3F740] vtkEGLRenderWindow.cxx:353   WARN| vtkEGLRenderWindow (0x8f287c0): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: pci id for fd 95: 10de:26b2, driver (null)

pci id for fd 96: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 95: 10de:26b2, driver (null)

pci id for fd 96: 10de:26b2, driv

/cwork/ns458/insula-paper/img/supplementary/operculum_pairs/LexicalDelay__D0079_LAI1-2__D0079_LFMI7-8__hga_coronal_pub.svg


libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card8: Permission denied

2026-10-20 21:51:38.494 (   2.188s) [    7FB5F6E3F740] vtkEGLRenderWindow.cxx:353   WARN| vtkEGLRenderWindow (0x108df530): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/c

/cwork/ns458/insula-paper/img/supplementary/operculum_pairs/LexicalDelay__D0079_LAI2-3__D0079_LFMI7-8__hga_coronal_pub.svg


libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card8: Permission denied

2026-10-20 21:51:39.538 (   3.232s) [    7FB5F6E3F740] vtkEGLRenderWindow.cxx:353   WARN| vtkEGLRenderWindow (0xb2848b0): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/ca

/cwork/ns458/insula-paper/img/supplementary/operculum_pairs/LexicalDelay__D0079_LAI3-4__D0079_LFMI7-8__hga_coronal_pub.svg


... (20 outputs left out)

libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card8: Permission denied

2026-10-20 21:51:49.862 (  13.556s) [    7FB5F6E3F740] vtkEGLRenderWindow.cxx:353   WARN| vtkEGLRenderWindow (0x121f9810): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/c

/cwork/ns458/insula-paper/img/supplementary/operculum_pairs/SentenceRep__D0064_LAI6-7__D0064_LAIF6-7__hga_coronal_pub.svg


libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card8: Permission denied

2026-10-20 21:51:50.738 (  14.432s) [    7FB5F6E3F740] vtkEGLRenderWindow.cxx:353   WARN| vtkEGLRenderWindow (0xf84bc70): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 96: 10de:26b2, driver (null)

pci id for fd 98: 10de:26b2, driver (null)
kmsro: driver missing
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/ca

/cwork/ns458/insula-paper/img/supplementary/operculum_pairs/SentenceRep__D0102_RFAI1-2__D0102_RFAI3-4__hga_coronal_pub.svg
drew 15
skipped 0
